# Act 1 — Three storage shapes, three different services

You are going to need to store data on AWS. The first decision is not which service — it is which **shape of access** you need. There are three categories, and they are not interchangeable.

**Object storage** is files served over HTTP, scaling to infinity, with no filesystem semantics. **Block storage** is a virtual disk attached to one machine. **File storage** is a network filesystem that many machines mount at the same time.

The choice between these three is the one that matters most. Inside each category, the AWS service that fits is largely obvious — S3 for object, EBS or Instance Store for block, EFS or FSx for file.

## The three categories at a glance

AWS storage splits cleanly along the access pattern. **Object storage** — S3 — serves files over HTTP and scales to any size. **Block storage** — EBS and Instance Store — attaches to a single EC2 instance as a virtual disk. **File storage** — EFS and the FSx family — exposes a network filesystem that many instances can mount at once. Pick the wrong category and nothing about the service will fit your workload; pick the right one and the specifics are mostly tuning.

# Act 2 — S3, the centre of gravity

S3 is the most-used storage service on AWS, by a wide margin. Almost every architecture touches it somewhere. It is the durable home for backups, the staging area for analytics, the publishing layer for static sites, the event source that fires Lambdas, the destination for cross-region replication, the input and output of every data pipeline.

What makes S3 work is the simplicity of its mental model. There are buckets and there are objects in those buckets, addressed by string keys. Every other feature — storage classes, security, lifecycle rules, replication, access patterns — bolts onto that minimal core. This act walks each of those features in the order you actually meet them when running a real bucket in production.

## S3 — Buckets and Objects

S3 stores **objects** in **buckets**. A bucket is a container with a globally unique name; an object is a file plus its metadata, addressed by a **key** that is just a string. The key `images/2024/photo.jpg` looks like a path, but S3 is flat — the slashes are part of the name. The console renders prefixes as folders for convenience; the underlying storage has no directories.

| Property | Detail |
|---|---|
| **Bucket name** | Globally unique across all AWS accounts; 3–63 chars, lowercase, no underscores |
| **Bucket region** | Data never leaves the region unless you explicitly replicate |
| **Object size** | 0 bytes to 5 TB |
| **Single PUT** | Max 5 GB — use multipart upload above that, recommended above 100 MB |
| **Metadata** | System (Content-Type, ETag) plus up to 2 KB of user-defined headers |

There is no limit on the number of objects in a bucket, and request capacity scales with prefix count. The flatness is what makes S3 cheap and infinitely scalable — every other property follows from it.

## Storage Classes

All classes share the same eleven-nines durability. They differ on availability, retrieval cost, and how long an object has to stay before deletion is free.

| Class | Availability | Min duration | Retrieval | Sweet spot |
|---|---|---|---|---|
| **Standard** | 99.99% | None | Instant | Hot data, default |
| **Intelligent-Tiering** | 99.9% | None | Instant (frequent/IA tiers); minutes for archive tiers | Unknown or shifting access patterns |
| **Standard-IA** | 99.9% | 30 days | Instant, with retrieval fee | Backups, monthly-accessed data |
| **One Zone-IA** | 99.5% | 30 days | Instant | Reproducible data only — single-AZ |
| **Glacier Instant Retrieval** | 99.9% | 90 days | Milliseconds | Archives read once a quarter |
| **Glacier Flexible Retrieval** | 99.99% | 90 days | Minutes to hours | Backups, DR |
| **Glacier Deep Archive** | 99.99% | 180 days | 12–48 hours | Long-term compliance |

A few traps are worth naming. **One Zone-IA** is 20% cheaper than Standard-IA because there's no cross-AZ replication — if the AZ is destroyed, the data is gone, so it's only for reproducible derivatives like thumbnails. **Glacier Instant Retrieval** behaves like Standard for reads but bills for retrieval — perfect when access is rare and immediate. **Intelligent-Tiering** charges a tiny per-object monitoring fee but has no retrieval cost, and it's the right answer when you genuinely don't know the access pattern. **Minimum duration charges** mean if you delete a Standard-IA object after a week, you still pay the full 30 days — so they're traps for hot data masquerading as cold.

## Security Model

Every bucket and every object is **private by default**. Access is granted through a small set of layered controls.

| Control | Scope | What it does |
|---|---|---|
| **Block Public Access** | Account or bucket | Overrides any policy that would make objects public — on by default for new buckets |
| **Bucket policy** | Bucket (resource-based) | JSON policy attached to the bucket; primary cross-account and conditional grant |
| **IAM policy** | Identity (user/role) | Same-account access from your principals |
| **ACLs** | Object or bucket (legacy) | Disabled by default since 2023; leave them off |

The evaluation rule that catches people: an **explicit deny** in any policy always wins, and **Block Public Access** runs before anything else — a bucket policy that grants `Principal: "*"` is silently ignored if BPA is enabled. That's the safety net behind every "we made the bucket public by accident" headline; AWS turned it on by default.

For cross-account access, both sides must allow. The source account's IAM policy must permit the action, and the bucket policy must permit the principal. Within a single account, either side allowing is enough.

## Encryption

Since January 2023, every new object is encrypted with **SSE-S3** by default — no configuration required. You override that choice when you need stronger guarantees about key management.

| Option | Keys managed by | When to choose |
|---|---|---|
| **SSE-S3** | AWS (opaque) | Default; sensible baseline |
| **SSE-KMS** | AWS KMS, your CMK | Audit trail in CloudTrail, per-key access control, rotation control |
| **SSE-C** | You — sent with every request | You insist AWS never sees the key |
| **Client-side** | You | End-to-end; AWS only sees ciphertext |

SSE-KMS is the right answer for regulated data — every decryption is a KMS API call that lands in CloudTrail, and IAM on the KMS key gives you a second access lever independent of the bucket policy. The catch is cost and throughput: KMS calls are billable and rate-limited. **S3 Bucket Keys** mitigate this by caching a bucket-level data key, cutting KMS calls by ~99%.

For transit, S3 supports both HTTP and HTTPS. To enforce HTTPS, add a bucket policy denying any request where `aws:SecureTransport` is `false` — this is standard hygiene on every production bucket.

## Versioning, Object Lock, and MFA Delete

**Versioning** preserves every PUT and every DELETE. A new upload creates a new version ID; a delete adds a **delete marker** that becomes the latest version, with the prior versions still retrievable underneath. You permanently remove an object by specifying its exact version ID.

Once versioning is enabled on a bucket it can only be **suspended**, never disabled — old versions outlive the setting. Storage costs accumulate across versions, so a versioned bucket almost always wants a lifecycle rule that expires non-current versions after some window.

**MFA Delete** adds an extra requirement — permanent deletion of a version, or any change to the versioning state, requires an MFA token. It can only be enabled by the root account and is reserved for the highest-value buckets.

**Object Lock** is the stronger guarantee: WORM (write-once-read-many) storage where objects cannot be modified or deleted for a retention period. It has to be enabled at bucket creation — you cannot retrofit it — and it forces versioning on.

| Retention mode | Who can override |
|---|---|
| **Governance** | Users with the `s3:BypassGovernanceRetention` permission |
| **Compliance** | Nobody, not even the root account, until the period expires |

A **legal hold** sits orthogonal to retention — it locks an object indefinitely, independent of any retention period, until a user with `s3:PutObjectLegalHold` removes it. Compliance mode plus a long retention plus legal holds is how regulated industries meet SEC 17a-4 and FINRA archival requirements.

## Lifecycle Policies

Lifecycle rules move objects between storage classes and eventually delete them, automatically, based on object age. A rule can apply to the whole bucket, to a prefix like `logs/`, to objects with specific tags, or to objects in a size range.

Transitions only flow **downward** in the cost ladder: Standard can move to Standard-IA, One Zone-IA, Intelligent-Tiering, or any Glacier tier; Glacier Instant can move to Glacier Flexible or Deep Archive. You cannot use lifecycle to promote an object back to a warmer class. The minimum age before transitioning out of Standard into an IA tier is 30 days.

The useful expiration actions are:

- **Expire current versions** after N days — the basic retention cap
- **Expire non-current versions** after N days — the cost lever that makes versioned buckets affordable
- **Delete expired delete markers** — housekeeping for delete markers that no longer hide any versions
- **Abort incomplete multipart uploads** after N days — kills orphaned parts that would otherwise bill forever

A typical production bucket runs four lifecycle rules in parallel — one transition ladder for hot data cooling off, plus the three housekeeping rules above.

## Replication

S3 asynchronously copies objects from a source bucket to a destination bucket — across regions (**CRR**) for DR, latency, and data residency, or within a region (**SRR**) for log aggregation and cross-account copies. Both source and destination must have versioning enabled, and a service role grants S3 permission to read from one and write to the other.

What replication does **not** do without extra configuration is the usual surprise:

- Existing objects are not backfilled — replication only catches new uploads. Use **S3 Batch Replication** to copy historical objects.
- Delete markers are not replicated by default — this is a feature, not a bug. It stops accidental deletes from cascading. You opt in if you want them replicated.
- Replication does not chain — A → B does not trigger B → C.
- Objects already in Glacier are skipped.
- SSE-KMS objects need explicit KMS key configuration on both sides.

**Replication Time Control (RTC)** is the SLA upgrade: 99.99% of new objects replicated within 15 minutes, with CloudWatch metrics for replication lag and pending counts. Without RTC, replication is best-effort and usually fast, but with no guarantee. RTC is the right choice when your RPO is contractual.

## Performance

S3 scales request rate **per prefix**: 3,500 writes per second and 5,500 reads per second per prefix, with no upper limit on the number of prefixes. You scale throughput horizontally by spreading objects across more prefixes. Old guidance about randomising key prefixes for hashing is obsolete — S3 partitions automatically.

For object size, three patterns matter:

- **Multipart upload** splits a large object into parts uploaded in parallel and reassembled server-side. Required above 5 GB; recommended above 100 MB. The big win is retry granularity — a failed part is retried alone rather than restarting the whole upload.
- **Byte-range fetches** are the read-side analogue — request specific byte ranges in parallel for faster downloads, or grab just a header without pulling the whole file.
- **Transfer Acceleration** routes uploads through the nearest CloudFront edge and then over the AWS backbone to the bucket. The win comes from skipping public-internet hops between distant continents — it's an upload optimisation for global users, not a CDN.

## Pre-signed URLs, Access Points, and Object Lambda

A **pre-signed URL** is a short-lived URL that grants temporary access to a specific object on behalf of an IAM identity. The recipient does not need AWS credentials — the signature in the URL carries the authorisation. The URL inherits whatever the signer is allowed to do, and expires in at most 7 days (and only that long if signed with long-lived credentials; STS-issued URLs are bounded by the session). The two big use cases are letting a user download a private object via a link, and letting a browser upload directly to S3 without round-tripping through your backend.

**S3 Access Points** are named endpoints attached to a bucket, each with its own access policy. Instead of one increasingly tangled bucket policy that tries to serve every team, you create one access point per use case — `analytics-ap` for read-only analytics, `app-ap` for the application's write path, `vpc-ap` for VPC-only access — each with a focused policy.

**S3 Object Lambda** sits in front of an access point and runs a Lambda function on the response before it reaches the caller. The Lambda can redact PII, resize an image, convert formats, or enrich data — without storing a transformed copy. The same source object can be served differently to different access points, each running its own transform.

## Event Notifications

S3 emits events when objects are created, removed, restored from Glacier, replicated, or transitioned by lifecycle. Events can be routed to SQS, SNS, Lambda, or EventBridge.

The direct destinations (SQS / SNS / Lambda) are simple but limited — one destination per event type with basic prefix and suffix filters. **EventBridge** receives every S3 event when enabled on the bucket and adds proper pattern matching, multiple targets per rule, archive and replay, and routing to any EventBridge target. The decision rule: direct destinations for the one-trigger-one-handler case; EventBridge as soon as the routing has any complexity to it.

## S3 Select, Batch Operations, and Storage Lens

Three S3 capabilities sit slightly outside the core storage model.

**S3 Select** runs SQL against the contents of a single object — CSV, JSON, or Parquet — server-side, returning only the matching rows. It cuts data transfer and processing cost dramatically when you only need a slice of a large file. The Glacier variant, Glacier Select, filters at the archive level before paying for a restore.

**S3 Batch Operations** runs a single action across billions of objects as one managed job: copy, invoke a Lambda per object, replace tags or ACLs, restore from Glacier, or run **Batch Replication** to backfill objects that existed before replication was configured. You feed it an S3 Inventory file as the manifest and it produces a completion report.

**S3 Storage Lens** is the org-wide analytics dashboard — 29+ metrics across every account and bucket in an Organization, with cost-optimisation recommendations (buckets without lifecycle, incomplete multipart uploads piling up, large non-current version stores). Metrics export to S3 for querying with Athena.

## Static Website Hosting

S3 can serve a static site directly — enable website hosting on the bucket, set an index document and error document, and access it on the `s3-website` endpoint. For anything production-grade, put **CloudFront** in front: it gives you a custom domain, HTTPS via an ACM certificate, global edge caching, and **Origin Access Control (OAC)** so the bucket stays private and only CloudFront can read from it. WAF sits naturally at the CloudFront edge as well. The bare S3 website endpoint is for quick internal sites; the CloudFront + OAC pairing is the production default.

In [ ]:
import boto3, json

s3 = boto3.client("s3", region_name="us-east-1")
bucket = "my-demo-bucket-unique-123"

s3.create_bucket(Bucket=bucket)

# Versioning + SSE-KMS with a Bucket Key (cuts KMS API cost)
s3.put_bucket_versioning(Bucket=bucket, VersioningConfiguration={"Status": "Enabled"})
s3.put_bucket_encryption(Bucket=bucket, ServerSideEncryptionConfiguration={
    "Rules": [{
        "ApplyServerSideEncryptionByDefault": {"SSEAlgorithm": "aws:kms"},
        "BucketKeyEnabled": True,
    }]
})

# Lifecycle: tier logs, expire old versions, abort stuck uploads
s3.put_bucket_lifecycle_configuration(Bucket=bucket, LifecycleConfiguration={"Rules": [
    {"ID": "tier-logs", "Status": "Enabled", "Filter": {"Prefix": "logs/"},
     "Transitions": [
         {"Days": 30,  "StorageClass": "STANDARD_IA"},
         {"Days": 90,  "StorageClass": "GLACIER"},
         {"Days": 365, "StorageClass": "DEEP_ARCHIVE"},
     ],
     "Expiration": {"Days": 2555}},
    {"ID": "expire-noncurrent", "Status": "Enabled", "Filter": {},
     "NoncurrentVersionExpiration": {"NoncurrentDays": 60},
     "ExpiredObjectDeleteMarker": True},
    {"ID": "abort-multipart", "Status": "Enabled", "Filter": {},
     "AbortIncompleteMultipartUpload": {"DaysAfterInitiation": 7}},
]})

# Enforce HTTPS via bucket policy
s3.put_bucket_policy(Bucket=bucket, Policy=json.dumps({
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Deny", "Principal": "*", "Action": "s3:*",
        "Resource": [f"arn:aws:s3:::{bucket}", f"arn:aws:s3:::{bucket}/*"],
        "Condition": {"Bool": {"aws:SecureTransport": "false"}},
    }],
}))

# Pre-signed URL — temporary download link, no AWS credentials needed on the recipient
url = s3.generate_presigned_url(
    "get_object", Params={"Bucket": bucket, "Key": "reports/q4.pdf"}, ExpiresIn=3600,
)

# Act 3 — EBS, a disk for one machine

S3 is the right place for files served over HTTP and accessed by name. But your EC2 instance still needs a disk — somewhere the operating system installs to, somewhere a database can perform block-level random I/O, somewhere your application reads and writes files the way every program has read and written files for the last fifty years.

That is the block-storage problem, and AWS gives you two answers. **EBS** is the network-attached, durable, snapshotable, detach-and-reattach option that almost everything uses. **Instance Store** is the local-NVMe, blazing-fast, but completely ephemeral option for workloads where speed matters more than durability. This act covers both.

## EBS — Block Storage for EC2

An **EBS volume** is a network-attached block device that an EC2 instance sees as a local disk. Because it's network-attached rather than physically inside the host, three properties follow:

- It can be **detached from one instance and reattached to another** in the same AZ
- It **persists independently of the instance** — survives stop/terminate unless Delete-on-Termination is set
- It pays a small network latency tax compared to local disks (negligible for everything except the most latency-sensitive workloads)

EBS volumes are **AZ-scoped** — a volume in `us-east-1a` only attaches to instances in `us-east-1a`. Crossing AZs or regions goes through a snapshot. Each attachment carries a **Delete on Termination** flag: on by default for the root volume, off by default for additional volumes.

Most volumes attach to one instance at a time. **Multi-Attach** is the exception — io1 and io2 volumes can attach to up to 16 instances in the same AZ simultaneously, but the application needs a **cluster-aware filesystem** like GFS2. A regular ext4 or XFS mounted by two writers will corrupt itself; Multi-Attach is for HA cluster software that coordinates writes explicitly.

## Volume Types

| Type | Class | Max IOPS | Max Throughput | Where it fits |
|---|---|---|---|---|
| **gp3** | General SSD | 16,000 | 1,000 MB/s | Default for almost everything |
| **gp2** | General SSD | 16,000 | 250 MB/s | Legacy — migrate to gp3 |
| **io2** | Provisioned IOPS SSD | 64,000 | 1,000 MB/s | I/O-intensive databases |
| **io2 Block Express** | Provisioned IOPS SSD | 256,000 | 4,000 MB/s | The largest databases (SAP HANA, big Oracle) |
| **st1** | Throughput HDD | 500 | 500 MB/s | Sequential big-data, log processing — not bootable |
| **sc1** | Cold HDD | 250 | 250 MB/s | Rarely-accessed bulk data — not bootable |

The practical decision tree is short. Almost all new workloads want **gp3** — 3,000 IOPS and 125 MB/s baseline regardless of volume size, with IOPS and throughput dialled up independently (gp2 ties IOPS to size at 3 IOPS/GB, which is why a 100 GB gp2 boots slower than a 100 GB gp3). It's also about 20% cheaper than gp2 for the same baseline.

Reach for **io2** when you need more than 16,000 IOPS or sustained microsecond-latency performance for a critical database. Prefer io2 over io1 unconditionally — same price, 100× more durable. **io2 Block Express** unlocks the upper end at 256,000 IOPS, available only on Nitro-based instances like r5b and x2idn.

**st1** and **sc1** are HDD-backed and cannot be boot volumes. st1 is the right pick for streaming sequential workloads — Kafka, log processing, Hadoop scans — where MB/s matters more than IOPS. sc1 is the cheapest tier, for archives you might want to mount occasionally.

## Snapshots

A snapshot is a point-in-time backup of a volume, stored in S3 behind the scenes (you don't see the bucket — AWS manages it). Snapshots are **incremental**: the first one copies every used block, every subsequent snapshot copies only blocks that changed since the prior. Restore creates a new EBS volume from the snapshot, in any AZ of the same region — or in another region after copying the snapshot across.

Three snapshot features extend the basic model:

- **EBS Snapshot Archive** is a separate tier that's about 75% cheaper for snapshots you keep for compliance but rarely restore. Restoration takes 24–72 hours.
- **Recycle Bin** holds deleted snapshots for a configured retention window (1–365 days) before permanent deletion — the safety net against fat-fingered cleanups.
- **Fast Snapshot Restore (FSR)** pre-warms a snapshot in specific AZs so volumes created from it have full performance immediately, with no lazy-loading latency on first-touch blocks. It's expensive — reserve it for DR exercises and rapid scaling from a golden snapshot.

Snapshots are also the path for **encrypting an existing unencrypted volume**: snapshot it, copy the snapshot with `Encrypted=true`, create a new volume from the encrypted copy, swap the attachment. There is no in-place encryption operation.

## Encryption

EBS encryption is AES-256 backed by KMS, applied transparently — data at rest, snapshots, and the traffic between the volume and the instance are all encrypted. The Nitro hardware does the work, so latency overhead is negligible. Snapshots of an encrypted volume are encrypted; volumes restored from an encrypted snapshot are encrypted.

The one configuration knob worth flipping early is **account-level default encryption** — when set, every new volume and every snapshot copy is automatically encrypted with your chosen KMS key. Existing unencrypted volumes are not retroactively encrypted, so enable this in fresh accounts before workloads land in them.

## Instance Store

Instance Store is **physically attached NVMe** on the EC2 host — the disk lives inside the server running the instance, with no network between them. The trade is brutal: millions of IOPS and sub-millisecond latency on one side, **complete ephemerality** on the other. Stop the instance, terminate it, or lose the underlying host, and the data is gone. A reboot keeps it (same host, instance stays running).

Instance Store is included in the instance price, sized fixed per instance type, and cannot be detached or resized. The use cases are workloads where speed matters more than durability and durability is provided elsewhere:

- Local caches (Redis-style state that rebuilds on startup)
- Scratch space for ML training and batch jobs
- Distributed systems that replicate at the application layer — Cassandra, Elasticsearch, Kafka — where node-local data loss is normal and the cluster heals itself

Storage-optimised instance families — `i3`, `i4i`, `i4g`, `d3`, `im4gn` — carry the largest Instance Store allocations.

The split with EBS is the durability axis: EBS for anything that has to survive the instance, Instance Store for anything that should be fast and is fine being lost.

In [ ]:
import boto3

ec2 = boto3.client("ec2", region_name="us-east-1")

# gp3 with IOPS and throughput dialled independently of size
vol = ec2.create_volume(
    AvailabilityZone="us-east-1a",
    VolumeType="gp3",
    Size=100,
    Iops=6000,         # baseline is 3,000
    Throughput=250,    # baseline is 125 MB/s
    Encrypted=True,
)

# Snapshot, then copy to another region with encryption — the DR pattern
snap = ec2.create_snapshot(VolumeId=vol["VolumeId"], Description="pre-deploy backup")
ec2_eu = boto3.client("ec2", region_name="eu-west-1")
ec2_eu.copy_snapshot(
    SourceRegion="us-east-1", SourceSnapshotId=snap["SnapshotId"],
    Description="DR copy", Encrypted=True,
)

# Online migration gp2 -> gp3 — no downtime, attachment stays live
ec2.modify_volume(VolumeId="vol-0abc123", VolumeType="gp3", Iops=3000, Throughput=125)

# Act 4 — When many machines need the same files

EBS attaches to one instance at a time. What happens when you have ten web servers behind a load balancer and they all need to see the same uploads directory? Or an ML training cluster that should all read from one shared dataset? Or a Windows application that expects an SMB share?

That is the file-storage problem — a network filesystem that many clients can mount simultaneously. AWS gives you two umbrella services. **EFS** is the Linux and NFS answer, regional and elastic. The **FSx family** is everything else — SMB for Windows, Lustre for HPC, NetApp ONTAP and OpenZFS for lift-and-shift cases. This act covers EFS in depth, then walks the FSx options.

## EFS — Elastic File System

EFS is a managed **NFS v4** filesystem that many Linux instances mount simultaneously — across AZs, across accounts, and even from on-premises over Direct Connect or VPN. Unlike EBS, it's **regional**: data is replicated across multiple AZs automatically, capacity grows and shrinks elastically, and you pay per GB stored with no provisioning.

Clients reach the filesystem through **mount targets** — one per AZ, each with a private IP in your VPC. Security groups on the mount target control which instances can connect. All mount targets resolve to the same filesystem, so a write through any mount target is immediately visible to every other.

EFS is Linux-only and POSIX-compliant — Windows clients need FSx for Windows File Server instead. The typical workloads are content management systems shared across web servers, Linux home directories, shared configuration or code drops across an Auto Scaling Group, and container volumes for ECS and EKS (via the EFS CSI driver).

## Performance and Throughput Modes

EFS has two orthogonal performance knobs.

**Performance mode** is set at creation and not changeable:

| Mode | Trait |
|---|---|
| **General Purpose** (default) | Lowest latency; right for almost everything — web serving, CMS, home dirs |
| **Max I/O** | Higher latency but scales to thousands of concurrent clients — large analytics, media pipelines |

**Throughput mode** is changeable any time:

| Mode | How throughput is determined |
|---|---|
| **Elastic** (default) | Auto-scales to demand; pay per GB transferred — best for spiky or unknown patterns |
| **Bursting** | Baseline scales with stored size (50 KB/s per GB); burst credits cover spikes |
| **Provisioned** | Fixed throughput independent of size — small dataset that needs high throughput |

Bursting is the original model and still fine when storage is large enough to give comfortable baseline. Elastic removes the planning entirely and is the right default for new filesystems.

## Storage Tiers and One Zone

EFS has its own internal tiering, parallel to S3's storage classes. Files move between tiers based on **last access time**, configured by a lifecycle policy on the filesystem.

| Tier | Trigger | Cost |
|---|---|---|
| **Standard** | Active files | Baseline |
| **Standard-IA** | Not accessed for N days | ~92% cheaper than Standard |
| **Archive** | Not accessed for longer N | ~95% cheaper than Standard |

An optional rule promotes a file back to Standard on access — useful when occasional reads should not pay the IA retrieval fee twice. The cold tiers have a retrieval cost on read, so they only pay off for genuinely cold files.

**EFS One Zone** is a single-AZ variant — about 47% cheaper than the multi-AZ default, suitable for dev/test, build caches, or data that's already backed up elsewhere. The trade-off is the obvious one: if the AZ fails, the filesystem is unavailable until it recovers, and a lost AZ means lost data.

## FSx — Specialised File Systems

FSx is a family of managed third-party filesystems, each purpose-built for a workload that EFS doesn't fit.

| Variant | Protocol | Where it fits |
|---|---|---|
| **FSx for Windows File Server** | SMB | Windows workloads that need NTFS and Active Directory |
| **FSx for Lustre** | Lustre / POSIX | HPC, ML training, media processing — extreme throughput |
| **FSx for NetApp ONTAP** | NFS + SMB + iSCSI | Multi-protocol enterprise, lift-and-shift from NetApp |
| **FSx for OpenZFS** | NFS | ZFS workloads, fast clones and snapshots |

The two everyone needs to know are **Windows File Server** (the answer whenever SMB or Active Directory enters the question) and **Lustre** (the answer for HPC and ML at scale). ONTAP and OpenZFS are lift-and-shift answers — you choose them because you already run NetApp or ZFS and don't want to rearchitect.

## Windows File Server and Lustre in More Detail

**FSx for Windows File Server** is a fully managed Windows filesystem running on Windows Server underneath. It speaks SMB, stores files in NTFS, integrates with Active Directory (self-managed or AWS Managed Microsoft AD), and supports DFS Namespaces for federating multiple filesystems under a single name. Multi-AZ deployment runs an active filesystem in one subnet and an automatically-promoted standby in another. Daily backups land in S3. Lift-and-shift Windows apps, SQL Server file shares, and Windows user home directories are the canonical workloads.

**FSx for Lustre** delivers up to hundreds of GB/s of throughput, millions of IOPS, and sub-millisecond latency for workloads that need to chew through huge datasets fast — ML training, genomics, financial modelling, seismic analysis, video processing. The S3 integration is the architecturally interesting piece: you link a Lustre filesystem to an S3 bucket, files appear in the filesystem on first access, and processed results can be exported back to S3. The data lives in S3 cheaply; Lustre is the high-performance compute-time mount.

Lustre has two deployment types. **Scratch** filesystems are temporary, not replicated, and the cheapest option — used for processing runs where the output goes back to S3 and the filesystem is torn down. **Persistent** filesystems replicate within an AZ and survive for long-running jobs or workloads where the filesystem is the primary store.

# Act 5 — The hybrid bridge

Everything so far assumes the storage and the compute both live in AWS. Real organizations are messier. There are on-premises file shares that applications still expect to be there. There are backup tools that write to iSCSI block volumes. There are physical tape libraries running compliance retention.

Migrating those workloads to AWS storage without rewriting the applications that depend on them is the hybrid-storage problem, and **AWS Storage Gateway** is the answer. This act covers the gateway flavours, then consolidates the file-storage decision across every option in the notebook.

## Storage Gateway — the Hybrid Bridge

Storage Gateway runs as a VM (or hardware appliance) in your on-premises data centre and presents AWS storage as local storage to on-prem systems. There are four flavours, distinguished by what they expose locally and what backs them in AWS.

| Gateway | Local protocol | AWS backing | Typical use |
|---|---|---|---|
| **S3 File Gateway** | NFS / SMB | S3 | On-prem file share that lands files directly in S3 |
| **FSx File Gateway** | SMB | FSx for Windows | Remote-office local cache for a central FSx |
| **Volume Gateway — Stored** | iSCSI | EBS snapshots in S3 | Full dataset on-prem, async backup to AWS |
| **Volume Gateway — Cached** | iSCSI | S3 | Hot data cached on-prem, primary in AWS |
| **Tape Gateway** | iSCSI VTL | S3 / Glacier | Replace physical tape libraries — backup software unchanged |

The two split decisions worth internalising. **S3 File Gateway vs Volume Gateway**: file (NFS/SMB) versus block (iSCSI) — same as S3 versus EBS, just with the gateway as a translator. **Stored vs Cached** Volume Gateway: where the full dataset lives. Stored keeps it locally with cloud backup (low-latency reads, minimal cloud dependency); Cached keeps it in S3 with only hot data on-prem (minimal local storage, more cloud dependency).

## Choosing Among the File Storage Options

The file storage options are the easiest place to pick wrong, because the differences are protocol-driven and the names don't tell you which one fits.

| Need | Service |
|---|---|
| Linux NFS, shared across many instances, multi-AZ | **EFS** |
| Windows SMB, NTFS, Active Directory | **FSx for Windows File Server** |
| Maximum throughput for HPC or ML training, optionally backed by S3 | **FSx for Lustre** |
| Multi-protocol (NFS + SMB + iSCSI), lift-and-shift from NetApp | **FSx for NetApp ONTAP** |
| ZFS features — instant clones, snapshots, point-in-time recovery | **FSx for OpenZFS** |
| On-prem clients with files landing in S3 | **S3 File Gateway** |
| On-prem clients with block storage, primary in AWS or on-prem | **Volume Gateway (Cached / Stored)** |
| Replace a physical tape library | **Tape Gateway** |

The operator-of-last-resort answer to most generic "shared file storage" questions on AWS is EFS for Linux and FSx for Windows. Everything else is a specialised tool for a recognisable shape of workload.

In [ ]:
import boto3, uuid

efs = boto3.client("efs", region_name="us-east-1")
fsx = boto3.client("fsx", region_name="us-east-1")

# EFS — elastic throughput, lifecycle into cold tiers based on last access
fs = efs.create_file_system(
    CreationToken=str(uuid.uuid4()),
    PerformanceMode="generalPurpose",
    ThroughputMode="elastic",
    Encrypted=True,
)
efs.put_lifecycle_configuration(
    FileSystemId=fs["FileSystemId"],
    LifecyclePolicies=[
        {"TransitionToIA": "AFTER_30_DAYS"},
        {"TransitionToArchive": "AFTER_90_DAYS"},
        {"TransitionToPrimaryStorageClass": "AFTER_1_ACCESS"},
    ],
)

# FSx for Lustre linked to S3 — ML training pattern
# Data lives cheaply in S3; mount Lustre during the run; results export back
fsx.create_file_system(
    FileSystemType="LUSTRE",
    StorageCapacity=1200,
    StorageType="SSD",
    SubnetIds=["subnet-0abc111"],
    SecurityGroupIds=["sg-0lustre789"],
    LustreConfiguration={
        "ImportPath": "s3://ml-training-data/",
        "ExportPath": "s3://ml-training-data/results",
        "DeploymentType": "SCRATCH_2",
        "AutoImportPolicy": "NEW_CHANGED_DELETED",
    },
)